In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
 
pd.set_option('display.max_colwidth', 80)
 
# ## Step 1: Build the labeled training dataset
# A synthetic but balanced set of `Flirt` and `Non_Flirt` example messages.
 
non_flirt_messages = [
    "Hi Mam", "Good morning", "Good evening", "How are you",
    "I have completed the MCQ python test", "I have some doubts in python interview questions",
    "Python debugging set1 Mam", "Can you share the class recording",
    "I submitted the assignment", "What time is the session today",
    "Thank you for the help", "I am facing an error in my code",
    "Please check my resume", "The meeting has been rescheduled",
    "I completed the SQL module", "Can we reschedule the call",
    "I will send the report by evening", "The server is down",
    "Let's discuss the project timeline", "Mam kindly send me the link to login",
    "I got an error while installing pandas", "Could you review my notebook",
    "The assignment deadline is tomorrow", "I have a doubt in the logistic regression topic",
    "Sure mam, I will complete it", "Ok mam", "Got it", "Noted, thank you",
    "I will join the session at 6pm", "Please share the attendance sheet",
    "Is the exam postponed", "I have uploaded the file to github",
    "Can you explain confusion matrix", "What is the difference between bagging and boosting",
    "I finished the data cleaning part", "The model accuracy is low, any suggestions",
    "I need help with the SQL query", "Please find the attached document",
    "The internet was down so I missed the class", "Can you resend the notes",
    "I will update you once it's done", "Thanks for the quick response",
    "Let me know if you need anything else", "I am working on the resume project",
    "The api call is returning an error", "Can we schedule a mock interview",
    "I have submitted the feedback form", "Please approve my leave request",
    "What is the syllabus for next week", "I will share the certificate once I get it",
    "The dataset has missing values, how do I handle them", "I am stuck at the deployment step",
    "Can you share the interview questions pdf", "I have completed all the modules",
    "The train and test split is done", "I used tfidf vectorizer for this task",
    "Random forest gave better accuracy than logistic regression", "I need to submit this by Friday",
    "Could you check if my code is correct", "The office is closed tomorrow",
    "I will call you after the meeting", "Please share the zoom link",
    "My laptop is not working properly", "I have a doubt regarding numpy arrays",
    "Thanks mam for the guidance", "I will revise this topic again",
    "Can you check my project structure", "The training loss is not decreasing",
    "I have a family function so I will be on leave", "Please confirm the interview time",
    "I updated the readme file", "The pull request is ready for review",
    "Can we do a doubt clearing session", "I will practice more sql queries",
]
 
flirt_messages = [
    "I hope I get to see you soon", "You make me feel special",
    "I like the way you look at me", "You are my favorite distraction",
    "I think we should spend more time together", "You have such a beautiful smile",
    "I can't stop thinking about you", "You look stunning today",
    "I miss you already", "Talking to you makes my day better",
    "You have the most charming personality", "I love our long conversations",
    "You are so cute when you laugh", "I get butterflies when you text me",
    "You are always on my mind", "I wish you were here with me right now",
    "Your voice is so soothing to listen to", "I can't help but smile when I see your name pop up",
    "You have the prettiest eyes", "Every time we talk, my heart races a little",
    "I'd love to take you out sometime", "You looked amazing in that photo",
    "I feel so comfortable when I'm around you", "Do you know how attractive you are",
    "I keep replaying our conversation in my head", "You are the highlight of my day",
    "I can't wait to see you again", "You have a way of making me blush",
    "I think about you more than I should", "Your smile is honestly unforgettable",
    "I love it when you tease me like that", "You always know how to make me laugh",
    "Being with you feels so natural", "I'd choose talking to you over anything else",
    "You're the reason I keep checking my phone", "I really like where this is going between us",
    "You make my heart skip a beat", "I can't stop staring at your pictures",
    "You're so charming, it's unfair", "I love the way you say my name",
    "Are you free this weekend? I'd love to see you", "You're honestly all I think about lately",
    "I feel a real connection with you", "You make even boring days feel exciting",
    "I saved your photo because you looked too good", "I love it when you call me late at night",
    "You are exactly my type", "I get nervous every time I message you",
    "I wish I could hold your hand right now", "You make butterflies feel like an understatement",
    "I really enjoy flirting with you", "You're cute, has anyone told you that today",
    "I keep smiling at my phone because of you", "Let's meet up, just the two of us",
    "You're impossible not to fall for", "I love how easily you make me blush",
    "You have no idea how much I like you", "I can't focus on work, I'm thinking about you",
    "You're the cutest person I've talked to in a while", "I love our late night chats, they feel special",
    "I wish every conversation with you never ended", "You make me feel butterflies just by texting",
    "I'd love to know more about you, over coffee maybe", "You're so easy to fall for",
    "I keep thinking of cute things to say to you", "Talking to you is the best part of my day",
    "You always leave me smiling like an idiot", "I really like you, more than a friend",
    "You're stuck in my head today, in a good way", "I love it when you send me good morning texts",
    "Every message from you makes me happy", "I think you're absolutely adorable",
    "I wish this conversation could go on forever", "You have a way of making me feel wanted",
]
 
data = ([(m, 'Non_Flirt') for m in non_flirt_messages] +
        [(m, 'Flirt') for m in flirt_messages])
 
dataset = pd.DataFrame(data, columns=['Chat', 'Label'])
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
print(dataset['Label'].value_counts())
dataset.head()
 
# ## Step 2: Train/test split
 
x = dataset['Chat']
y = dataset['Label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', len(x_train), '| Test size:', len(x_test))
 
# ## Step 3: Vectorization (CountVectorizer & TF-IDF)
 
count_vectorizer = CountVectorizer(stop_words='english')
count_train = count_vectorizer.fit_transform(x_train)
count_test = count_vectorizer.transform(x_test)
 
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_test = tfidf_vectorizer.transform(x_test)
 
# ## Step 4: Train & compare models
# Logistic Regression, Naive Bayes, SVM, and Random Forest — each run on both vectorizers.
 
vectorized_data = {
    'Countvectorizer': (count_train, count_test),
    'Tfidfvectorizer': (tfidf_train, tfidf_test),
}
 
models = {
    'LR': LogisticRegression(max_iter=1000),
    'NB': MultinomialNB(),
    'SVC': SVC(),
    'RF': RandomForestClassifier(random_state=42),
}
 
results_list = []
trained_models = {}
 
for vec_name, (train_matrix, test_matrix) in vectorized_data.items():
    for model_name, model in models.items():
        model.fit(train_matrix, y_train)
        pred = model.predict(test_matrix)
        acc = accuracy_score(y_test, pred)
        results_list.append({'Vectorizer': vec_name, 'Model': model_name, 'Accuracy_score': acc})
        trained_models[(vec_name, model_name)] = model
 
results = pd.DataFrame(results_list)
results
 
# ## Step 5: Pick the best vectorizer + model, retrain on the full dataset
 
best_row = results.sort_values('Accuracy_score', ascending=False).iloc[0]
best_vec_name, best_model_name = best_row['Vectorizer'], best_row['Model']
print(f"Best combo: {best_vec_name} + {best_model_name}  (accuracy={best_row['Accuracy_score']:.2f})")
 
final_vectorizer = CountVectorizer(stop_words='english') if best_vec_name == 'Countvectorizer' else TfidfVectorizer(stop_words='english')
final_train_matrix = final_vectorizer.fit_transform(x)  # fit on ALL labeled data for deployment
 
model_lookup = {
    'LR': LogisticRegression(max_iter=1000),
    'NB': MultinomialNB(),
    'SVC': SVC(),
    'RF': RandomForestClassifier(random_state=42),
}
final_model = model_lookup[best_model_name]
final_model.fit(final_train_matrix, y)
 
print(classification_report(y_test, trained_models[(best_vec_name, best_model_name)].predict(vectorized_data[best_vec_name][1])))
 
# Step 6: Parse a real WhatsApp chat export
# Reads the raw exported `.txt` file (e.g. `MeandRuby.txt`) into a clean `DataFrame` with `Date`, `Time`, `Sender`, `Chat` columns. Sender names are anonymized so no real names appear in the output.
# > Update `CHAT_FILE_PATH` and `NAME_MAP` below to match your own export.
 
CHAT_FILE_PATH = 'MeandRuby.txt'  # <-- place the exported .txt file in the same folder as this notebook
 
# Map real sender names -> anonymized labels. Adjust to match the names in YOUR export.
NAME_MAP = {
    'Kokila Venkatesh': 'You',
    'Rubhini Hope': 'Contact A',
}
 
line_pattern = re.compile(r'^(\d{1,2}/\d{1,2}/\d{2}),\s(\d{1,2}:\d{2}\s?[ap]m)\s-\s(.*)$')
 
def parse_whatsapp_chat(path, name_map=None):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            match = line_pattern.match(line)
            if match:
                date, time, rest = match.groups()
                if ': ' in rest:
                    sender, message = rest.split(': ', 1)
                    rows.append({'Date': date, 'Time': time, 'Sender': sender, 'Chat': message})
                else:
                    # system message (encryption notice, group change, etc.) - no sender
                    rows.append({'Date': date, 'Time': time, 'Sender': None, 'Chat': rest})
            else:
                # continuation of the previous multi-line message
                if rows and line.strip():
                    rows[-1]['Chat'] += ' ' + line.strip()
 
    chat_df = pd.DataFrame(rows)
    chat_df = chat_df.dropna(subset=['Sender'])  # drop system messages
    chat_df = chat_df[~chat_df['Chat'].str.contains(
        r'<Media omitted>|You deleted this message|This message was deleted', na=False, regex=True)]
    chat_df = chat_df[chat_df['Chat'].str.strip() != '']
    if name_map:
        chat_df['Sender'] = chat_df['Sender'].replace(name_map)
    return chat_df.reset_index(drop=True)
 
whatsapp_dataset = parse_whatsapp_chat(CHAT_FILE_PATH, NAME_MAP)
print('Total messages parsed:', len(whatsapp_dataset))
whatsapp_dataset.head(10)
 
# ## Step 7: Predict Flirt / Non_Flirt on the real chat
 
whatsapp_vector = final_vectorizer.transform(whatsapp_dataset['Chat'].astype(str))
whatsapp_dataset['Flirt_prediction'] = final_model.predict(whatsapp_vector)
whatsapp_dataset
 
whatsapp_dataset['Flirt_prediction'].value_counts()
 
whatsapp_dataset.groupby(['Sender', 'Flirt_prediction']).size().unstack(fill_value=0)
 

Label
Flirt        74
Non_Flirt    74
Name: count, dtype: int64
Train size: 118 | Test size: 30
Best combo: Countvectorizer + LR  (accuracy=0.93)
              precision    recall  f1-score   support

       Flirt       1.00      0.87      0.93        15
   Non_Flirt       0.88      1.00      0.94        15

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

Total messages parsed: 571


Flirt_prediction,Flirt,Non_Flirt
Sender,,
Contact A,15,187
You,15,354
